# 04 - Clustering & Insights

This notebook performs PCA dimensional reduction and clustering analysis on player statistics.

**Prerequisites**: Complete previous notebooks or run scripts up to `scripts/05_pca_clustering.py`


## Setup & Data Loading

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import modules
from src.utils import DataManager

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("✓ Modules imported successfully")


In [ ]:
# Initialize utilities
manager = DataManager()

# Load cleaned stats
try:
    df_clean = manager.load_pickle('data/processed/players_stats_clean.pkl')
    print(f"✓ Loaded cleaned stats: {df_clean.shape}")
except FileNotFoundError:
    print("⚠ Cleaned stats not found")
    df_clean = None

# Try to load PCA results
try:
    pca_clusters = pd.read_csv('data/processed/pca_clusters.csv')
    print(f"✓ Loaded PCA clusters: {pca_clusters.shape}")
except FileNotFoundError:
    print("⚠ PCA clusters not found yet")
    pca_clusters = None


## Step 1: Prepare Data for PCA

In [ ]:
# Variance explained plot
if df_clean is not None and 'pca' in locals():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    
    # Individual variance
    ax1.bar(range(1, min(len(pca.explained_variance_ratio_) + 1, 21)), 
            pca.explained_variance_ratio_[:20],
            alpha=0.7, color='steelblue')
    ax1.set_xlabel('Principal Component')
    ax1.set_ylabel('Explained Variance Ratio')
    ax1.set_title('Variance Explained by Component')
    ax1.grid(axis='y', alpha=0.3)
    
    # Cumulative variance
    cumsum_var = np.cumsum(pca.explained_variance_ratio_)
    ax2.plot(range(1, len(cumsum_var) + 1), cumsum_var, 'b-o', alpha=0.7)
    ax2.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
    ax2.set_xlabel('Number of Components')
    ax2.set_ylabel('Cumulative Explained Variance')
    ax2.set_title('Cumulative Variance Explained')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Visualization complete")


## Step 3: Visualize PCA Results

In [ ]:
# Apply PCA
if df_clean is not None and 'df_scaled' in locals():
    # Keep 95% of variance
    pca = PCA(n_components=0.95)
    pca_result = pca.fit_transform(df_scaled)
    
    print(f"✓ PCA Results:")
    print(f"  Original features: {df_scaled.shape[1]}")
    print(f"  PCA components: {pca_result.shape[1]}")
    print(f"  Variance explained: {pca.explained_variance_ratio_.sum():.2%}")
    print(f"\nExplained variance by component (top 10):")
    for i, var in enumerate(pca.explained_variance_ratio_[:10]):
        print(f"  PC{i+1}: {var:.2%}")
    
    # Create PCA dataframe
    pca_df = pd.DataFrame(
        pca_result,
        columns=[f'PC{i+1}' for i in range(pca_result.shape[1])]
    )
    
    # Add player identifier if available
    if 'wyId' in df_clean.columns:
        pca_df['wyId'] = df_clean['wyId'].values
        
    print(f"\n✓ PCA DataFrame shape: {pca_df.shape}")


## Step 2: Apply PCA

In [ ]:
# Prepare numeric features for PCA
if df_clean is not None:
    # Select only numeric columns
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    df_numeric = df_clean[numeric_cols].copy()
    
    print(f"✓ Numeric features: {len(numeric_cols)}")
    print(f"  Features: {numeric_cols[:10]}...")  # Show first 10
    print(f"\nDataset shape: {df_numeric.shape}")
    
    # Standardize the features
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(
        scaler.fit_transform(df_numeric),
        columns=numeric_cols,
        index=df_numeric.index
    )
    
    print(f"✓ Standardized data:")
    print(f"  Mean: {df_scaled.mean().mean():.4f}")
    print(f"  Std:  {df_scaled.std().mean():.4f}")


## Step 4: Analysis & Summary

The next notebook (`05_scoring_comparison.ipynb`) will use these PCA components for player position scoring.
